# Boosting Models

In this notebook, we compare three boosting regression methods for predicting the four mechanical properties used to assess weld quality: AdaBoost, Gradient Boosting, and Histogram Gradient Boosting. Each target is modeled separately using the same predefined cross-validation folds and evaluation protocol.

## Main observations

- **AdaBoost:** Even after hyperparameter tuning, AdaBoost performs substantially worse than the other boosting methods for all four regression targets and for the final conformity classification.

- **Gradient Boosting:** Tuned Gradient Boosting gives the best MAE for yield strength and UTS. It also provides the best overall conformity classification, with an F1-score of approximately 0.86 and a recall of approximately 0.81 for non-conforming welds.

- **Histogram Gradient Boosting:** This model is competitive with Gradient Boosting and gives the best MAE for elongation and Charpy toughness. It is particularly effective for Charpy toughness prediction.

- **Final comparison:** Gradient Boosting is retained as the strongest overall boosting approach because it provides the best final non-conformity classification performance, while Histogram Gradient Boosting performs better on some individual regression targets.

- **Limits:** Performance varies strongly between alloy families. The CrMo families contain much fewer observations, so their family-specific metrics are less stable and should be interpreted with caution.

## Operations in order

1. Load the training dataset using `load_train()`.
2. Evaluate a default Gradient Boosting model.
3. Tune Gradient Boosting hyperparameters using `GridSearchCV`.
4. Evaluate the tuned Gradient Boosting models with `evaluate()`.
5. Evaluate and tune AdaBoost.
6. Evaluate and tune Histogram Gradient Boosting.
7. Refine the Histogram Gradient Boosting search around the best parameter region.
8. Retain the best Histogram Gradient Boosting configuration for each target.
9. Save the tuned model results to the shared `results.csv`.
10. Compare all boosting approaches using MAE, RMSE, F1-score, and recall for non-conforming welds.

## 1. Setup and Data Loading

We use the shared evaluation functions of the project so that the boosting models are evaluated with exactly the same data, folds, and metrics as the other models.

In [6]:
import pandas as pd
import numpy as np

from helpers.utils import (
    TARGETS,
    load_train,
    get_xy,
    build_pipeline,
    evaluate,
    save_results
)

In [7]:
train = load_train()

print("Training set shape:", train.shape)
print("Targets:", TARGETS)
train.head()

Training set shape: (1332, 40)
Targets: ['yield_strength_MPa', 'uts_MPa', 'elongation_pct', 'charpy_toughness_J']


,C,Si,Mn,S,P,Ni,Cr,Mo,V,O_ppm,...,weld_type_GMAA,weld_type_GTAA,weld_type_MMA,weld_type_SA,weld_type_SAA,weld_type_TSA,AC_DC_unknown,input_group,family,fold
0,0.037,0.30,0.65,0.008,0.012,0.0,0.0,0.0,0.0,423.0,...,0,0,1,0,0,0,0,12,C-Mn,0
1,0.037,0.30,0.65,0.008,0.012,0.0,0.0,0.0,0.0,423.0,...,0,0,1,0,0,0,0,12,C-Mn,0
2,0.037,0.30,0.65,0.008,0.012,0.0,0.0,0.0,0.0,423.0,...,0,0,1,0,0,0,0,12,C-Mn,0
3,0.037,0.31,1.03,0.007,0.014,0.0,0.0,0.0,0.0,423.0,...,0,0,1,0,0,0,0,14,C-Mn,2
4,0.037,0.31,1.03,0.007,0.014,0.0,0.0,0.0,0.0,423.0,...,0,0,1,0,0,0,0,14,C-Mn,2


## 2. First Gradient Boosting Model

We first evaluate a default Gradient Boosting Regressor without hyperparameter tuning. This gives us a first reference result before optimization.

In [8]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    random_state=42
)

In [9]:
results_gb = evaluate(
    "Gradient Boosting",
    gb_model,
    train
)

results_gb

,model,target,family,n,MAE,RMSE,MAE_std,RMSE_std,n_clf,n_nc,F1_nc,recall_nc
0,Gradient Boosting,yield_strength_MPa,all,620,33.553538,47.883826,3.961933,9.478421,619,102,0.722892,0.588235
1,Gradient Boosting,yield_strength_MPa,C-Mn,552,31.964207,43.171840,1.937524,4.649855,552,102,0.722892,0.588235
2,Gradient Boosting,yield_strength_MPa,CrMo2,30,54.163566,88.137258,35.554997,48.974304,29,0,0.000000,0.000000
3,Gradient Boosting,yield_strength_MPa,CrMo91,38,40.369588,64.829026,20.814640,33.675743,38,0,0.000000,0.000000
4,Gradient Boosting,uts_MPa,all,596,29.384100,42.882317,4.538680,9.569167,595,198,0.705167,0.585859
5,Gradient Boosting,uts_MPa,C-Mn,531,27.013004,36.064787,2.182040,3.478345,531,198,0.705167,0.585859
6,Gradient Boosting,uts_MPa,CrMo2,34,55.362145,81.742274,27.417885,38.721959,33,0,0.000000,0.000000
7,Gradient Boosting,uts_MPa,CrMo91,31,41.506628,75.806196,23.282855,44.355052,31,0,0.000000,0.000000
8,Gradient Boosting,elongation_pct,all,558,2.044942,2.591623,0.129782,0.162669,557,48,0.578947,0.458333
9,Gradient Boosting,elongation_pct,C-Mn,497,2.036592,2.579255,0.132172,0.184322,497,37,0.634921,0.540541


## 3. Hyperparameter Tuning

The default Gradient Boosting model provides a first reference. We now tune its main hyperparameters using the predefined cross-validation folds. We start with yield strength before applying the same procedure to the other targets.

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor

target = "yield_strength_MPa"

X, y, cv = get_xy(train, target)

print("Number of observations:", len(y))
print("Number of features:", X.shape[1])

Number of observations: 620
Number of features: 30


In [13]:
gb = GradientBoostingRegressor(random_state=42)

pipeline = build_pipeline(gb)

param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_depth": [1, 2, 3]
}

In [14]:
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1
)

grid.fit(X, y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.03, 0.05, ...], 'model__max_depth': [1, 2, ...], 'model__n_estimators': [50, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","PredefinedSpl..., ..., 1, 3]))"
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the 

In [15]:
print("Best parameters:")
print(grid.best_params_)

print("\nBest cross-validation MAE:")
print(-grid.best_score_)

Best parameters:
{'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}

Best cross-validation MAE:
30.75423110615344


### 3.1 Tuning the Remaining Targets

The same hyperparameter search is now applied to UTS, elongation, and Charpy toughness. Each target is tuned separately because the best Gradient Boosting configuration may differ from one mechanical property to another.

In [16]:
targets_to_tune = [
    "uts_MPa",
    "elongation_pct",
    "charpy_toughness_J"
]

best_models = {
    "yield_strength_MPa": grid.best_estimator_
}

best_params = {
    "yield_strength_MPa": grid.best_params_
}

best_mae = {
    "yield_strength_MPa": -grid.best_score_
}

for target in targets_to_tune:
    X, y, cv = get_xy(train, target)

    model = GradientBoostingRegressor(random_state=42)
    pipeline = build_pipeline(model)

    grid_target = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1
    )

    grid_target.fit(X, y)

    best_models[target] = grid_target.best_estimator_
    best_params[target] = grid_target.best_params_
    best_mae[target] = -grid_target.best_score_

    print(f"\nTarget: {target}")
    print("Best parameters:", grid_target.best_params_)
    print("Best CV MAE:", -grid_target.best_score_)


Target: uts_MPa
Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}
Best CV MAE: 26.743461545823486

Target: elongation_pct
Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}
Best CV MAE: 1.9730790842199195

Target: charpy_toughness_J
Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}
Best CV MAE: 21.39376458558878


### 3.2 Refined Hyperparameter Search

The first grid search selected the upper boundary of the tested values for all four targets. This suggests that better configurations may exist outside the initial search range. We therefore perform a second search with a wider range around the best parameters.

In [17]:
refined_param_grid = {
    "model__n_estimators": [200, 300, 400],
    "model__learning_rate": [0.05, 0.1, 0.15],
    "model__max_depth": [2, 3, 4]
}

In [18]:
refined_best_models = {}
refined_best_params = {}
refined_best_mae = {}

for target in TARGETS:
    X, y, cv = get_xy(train, target)

    model = GradientBoostingRegressor(random_state=42)
    pipeline = build_pipeline(model)

    grid_refined = GridSearchCV(
        estimator=pipeline,
        param_grid=refined_param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1
    )

    grid_refined.fit(X, y)

    refined_best_models[target] = grid_refined.best_estimator_
    refined_best_params[target] = grid_refined.best_params_
    refined_best_mae[target] = -grid_refined.best_score_

    print(f"\nTarget: {target}")
    print("Best parameters:", grid_refined.best_params_)
    print("Best CV MAE:", -grid_refined.best_score_)


Target: yield_strength_MPa
Best parameters: {'model__learning_rate': 0.15, 'model__max_depth': 3, 'model__n_estimators': 400}
Best CV MAE: 28.575805900378686

Target: uts_MPa
Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 4, 'model__n_estimators': 400}
Best CV MAE: 25.213287271580818

Target: elongation_pct
Best parameters: {'model__learning_rate': 0.15, 'model__max_depth': 3, 'model__n_estimators': 300}
Best CV MAE: 1.9489772053200418

Target: charpy_toughness_J
Best parameters: {'model__learning_rate': 0.15, 'model__max_depth': 4, 'model__n_estimators': 400}
Best CV MAE: 19.63426922482558


### 3.3 Evaluation of the Tuned Gradient Boosting Models

The best Gradient Boosting configuration found for each target is now evaluated using the shared evaluation protocol. This allows direct comparison with the other supervised models using the same folds and metrics.

In [19]:
results_gb_tuned = evaluate(
    "Gradient Boosting Tuned",
    refined_best_models,
    train
)

results_gb_tuned

,model,target,family,n,MAE,RMSE,MAE_std,RMSE_std,n_clf,n_nc,F1_nc,recall_nc
0,Gradient Boosting Tuned,yield_strength_MPa,all,620,28.624816,41.533783,2.254498,6.421086,619,102,0.804348,0.725490
1,Gradient Boosting Tuned,yield_strength_MPa,C-Mn,552,26.893670,37.468254,0.962043,3.069723,552,102,0.804348,0.725490
2,Gradient Boosting Tuned,yield_strength_MPa,CrMo2,30,45.732721,71.327432,27.097675,33.838299,29,0,0.000000,0.000000
3,Gradient Boosting Tuned,yield_strength_MPa,CrMo91,38,40.265760,61.123222,19.664967,31.582022,38,0,0.000000,0.000000
4,Gradient Boosting Tuned,uts_MPa,all,596,25.279685,40.828429,4.229286,10.764204,595,198,0.763006,0.666667
5,Gradient Boosting Tuned,uts_MPa,C-Mn,531,23.224428,33.315250,2.302551,4.567728,531,198,0.763006,0.666667
6,Gradient Boosting Tuned,uts_MPa,CrMo2,34,40.705895,77.427485,30.848760,47.949850,33,0,0.000000,0.000000
7,Gradient Boosting Tuned,uts_MPa,CrMo91,31,43.565177,80.385758,26.960100,49.399641,31,0,0.000000,0.000000
8,Gradient Boosting Tuned,elongation_pct,all,558,1.951232,2.553755,0.208488,0.245324,557,48,0.585366,0.500000
9,Gradient Boosting Tuned,elongation_pct,C-Mn,497,1.936659,2.528157,0.215620,0.260422,497,37,0.606061,0.540541


### 3.4 Saving the Tuned Gradient Boosting Results

The tuned Gradient Boosting results are added to the shared results table so that they can be compared with the other model families.


In [20]:
all_results = save_results(results_gb_tuned)

all_results[
    all_results["model"] == "Gradient Boosting Tuned"
]

,model,target,family,n,MAE,RMSE,MAE_std,RMSE_std,n_clf,n_nc,F1_nc,recall_nc
60,Gradient Boosting Tuned,yield_strength_MPa,all,620,28.624816,41.533783,2.254498,6.421086,619,102,0.804348,0.725490
61,Gradient Boosting Tuned,yield_strength_MPa,C-Mn,552,26.893670,37.468254,0.962043,3.069723,552,102,0.804348,0.725490
62,Gradient Boosting Tuned,yield_strength_MPa,CrMo2,30,45.732721,71.327432,27.097675,33.838299,29,0,0.000000,0.000000
63,Gradient Boosting Tuned,yield_strength_MPa,CrMo91,38,40.265760,61.123222,19.664967,31.582022,38,0,0.000000,0.000000
64,Gradient Boosting Tuned,uts_MPa,all,596,25.279685,40.828429,4.229286,10.764204,595,198,0.763006,0.666667
65,Gradient Boosting Tuned,uts_MPa,C-Mn,531,23.224428,33.315250,2.302551,4.567728,531,198,0.763006,0.666667
66,Gradient Boosting Tuned,uts_MPa,CrMo2,34,40.705895,77.427485,30.848760,47.949850,33,0,0.000000,0.000000
67,Gradient Boosting Tuned,uts_MPa,CrMo91,31,43.565177,80.385758,26.960100,49.399641,31,0,0.000000,0.000000
68,Gradient Boosting Tuned,elongation_pct,all,558,1.951232,2.553755,0.208488,0.245324,557,48,0.585366,0.500000
69,Gradient Boosting Tuned,elongation_pct,C-Mn,497,1.936659,2.528157,0.215620,0.260422,497,37,0.606061,0.540541


## 4. AdaBoost

We now evaluate AdaBoost as a second boosting method. We first test the default model using the same shared evaluation protocol before tuning its hyperparameters.

In [21]:
from sklearn.ensemble import AdaBoostRegressor

ada_model = AdaBoostRegressor(
    random_state=42
)

results_ada = evaluate(
    "AdaBoost",
    ada_model,
    train
)

results_ada

,model,target,family,n,MAE,RMSE,MAE_std,RMSE_std,n_clf,n_nc,F1_nc,recall_nc
0,AdaBoost,yield_strength_MPa,all,620,46.418203,59.524652,3.013876,5.650064,619,102,0.019417,0.009804
1,AdaBoost,yield_strength_MPa,C-Mn,552,44.621864,55.749512,2.293851,2.812984,552,102,0.019417,0.009804
2,AdaBoost,yield_strength_MPa,CrMo2,30,56.097474,79.954562,33.432083,37.902152,29,0,0.000000,0.000000
3,AdaBoost,yield_strength_MPa,CrMo91,38,64.870860,87.264233,21.853349,32.979528,38,0,0.000000,0.000000
4,AdaBoost,uts_MPa,all,596,42.051228,53.667369,2.734761,7.325123,595,198,0.558621,0.409091
5,AdaBoost,uts_MPa,C-Mn,531,40.921098,49.296340,1.681499,2.261252,531,198,0.558621,0.409091
6,AdaBoost,uts_MPa,CrMo2,34,56.578371,84.193782,34.456991,42.777016,33,0,0.000000,0.000000
7,AdaBoost,uts_MPa,CrMo91,31,45.476271,77.288818,16.986593,40.005913,31,0,0.000000,0.000000
8,AdaBoost,elongation_pct,all,558,2.511611,3.148187,0.096397,0.128832,557,48,0.206897,0.125000
9,AdaBoost,elongation_pct,C-Mn,497,2.526797,3.159230,0.149138,0.191908,497,37,0.222222,0.135135


### 4.1 AdaBoost Hyperparameter Tuning

The default AdaBoost model performs significantly worse than Gradient Boosting. We therefore tune its main hyperparameters to check whether its performance can be improved before making a final comparison.

In [22]:
ada_param_grid = {
    "model__n_estimators": [50, 100, 200, 300],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
    "model__loss": ["linear", "square", "exponential"]
}

In [23]:
ada_best_models = {}
ada_best_params = {}
ada_best_mae = {}

for target in TARGETS:
    X, y, cv = get_xy(train, target)

    model = AdaBoostRegressor(random_state=42)
    pipeline = build_pipeline(model)

    grid_ada = GridSearchCV(
        estimator=pipeline,
        param_grid=ada_param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1
    )

    grid_ada.fit(X, y)

    ada_best_models[target] = grid_ada.best_estimator_
    ada_best_params[target] = grid_ada.best_params_
    ada_best_mae[target] = -grid_ada.best_score_

    print(f"\nTarget: {target}")
    print("Best parameters:", grid_ada.best_params_)
    print("Best CV MAE:", -grid_ada.best_score_)


Target: yield_strength_MPa
Best parameters: {'model__learning_rate': 1.0, 'model__loss': 'square', 'model__n_estimators': 300}
Best CV MAE: 43.717401221048725

Target: uts_MPa
Best parameters: {'model__learning_rate': 1.0, 'model__loss': 'square', 'model__n_estimators': 300}
Best CV MAE: 40.17335924759443

Target: elongation_pct
Best parameters: {'model__learning_rate': 1.0, 'model__loss': 'exponential', 'model__n_estimators': 300}
Best CV MAE: 2.413238629141563

Target: charpy_toughness_J
Best parameters: {'model__learning_rate': 0.01, 'model__loss': 'linear', 'model__n_estimators': 200}
Best CV MAE: 27.3842067316682


### 4.2 Evaluation of the Tuned AdaBoost Models

The best AdaBoost configuration found for each target is now evaluated using the shared evaluation protocol. This allows a direct comparison with Gradient Boosting and the other supervised models.

In [24]:
results_ada_tuned = evaluate(
    "AdaBoost Tuned",
    ada_best_models,
    train
)

results_ada_tuned

,model,target,family,n,MAE,RMSE,MAE_std,RMSE_std,n_clf,n_nc,F1_nc,recall_nc
0,AdaBoost Tuned,yield_strength_MPa,all,620,43.794743,57.894196,3.277517,6.491213,619,102,0.019417,0.009804
1,AdaBoost Tuned,yield_strength_MPa,C-Mn,552,42.013477,52.768571,2.612436,3.095387,552,102,0.019417,0.009804
2,AdaBoost Tuned,yield_strength_MPa,CrMo2,30,53.334205,81.642226,30.711648,42.112274,29,0,0.000000,0.000000
3,AdaBoost Tuned,yield_strength_MPa,CrMo91,38,62.138820,94.737552,33.812816,52.052692,38,0,0.000000,0.000000
4,AdaBoost Tuned,uts_MPa,all,596,40.214293,51.496811,3.038942,6.881391,595,198,0.579310,0.424242
5,AdaBoost Tuned,uts_MPa,C-Mn,531,39.149368,47.231626,1.506611,2.123658,531,198,0.579310,0.424242
6,AdaBoost Tuned,uts_MPa,CrMo2,34,51.342951,78.832453,31.041965,38.554373,33,0,0.000000,0.000000
7,AdaBoost Tuned,uts_MPa,CrMo91,31,46.249806,77.184630,16.291828,36.505299,31,0,0.000000,0.000000
8,AdaBoost Tuned,elongation_pct,all,558,2.414214,3.026609,0.130173,0.126526,557,48,0.181818,0.104167
9,AdaBoost Tuned,elongation_pct,C-Mn,497,2.435078,3.045255,0.190162,0.194951,497,37,0.186047,0.108108


### 4.3 Saving AdaBoost Results

The tuned AdaBoost results are saved in the shared results table. Although tuning improves some regression metrics, AdaBoost remains clearly less effective than Gradient Boosting for this dataset.

In [25]:
all_results = save_results(results_ada_tuned)

all_results[
    all_results["model"].isin([
        "Gradient Boosting Tuned",
        "AdaBoost Tuned"
    ])
]

,model,target,family,n,MAE,RMSE,MAE_std,RMSE_std,n_clf,n_nc,F1_nc,recall_nc
0,AdaBoost Tuned,yield_strength_MPa,all,620,43.794743,57.894196,3.277517,6.491213,619,102,0.019417,0.009804
1,AdaBoost Tuned,yield_strength_MPa,C-Mn,552,42.013477,52.768571,2.612436,3.095387,552,102,0.019417,0.009804
2,AdaBoost Tuned,yield_strength_MPa,CrMo2,30,53.334205,81.642226,30.711648,42.112274,29,0,0.000000,0.000000
3,AdaBoost Tuned,yield_strength_MPa,CrMo91,38,62.138820,94.737552,33.812816,52.052692,38,0,0.000000,0.000000
4,AdaBoost Tuned,uts_MPa,all,596,40.214293,51.496811,3.038942,6.881391,595,198,0.579310,0.424242
5,AdaBoost Tuned,uts_MPa,C-Mn,531,39.149368,47.231626,1.506611,2.123658,531,198,0.579310,0.424242
6,AdaBoost Tuned,uts_MPa,CrMo2,34,51.342951,78.832453,31.041965,38.554373,33,0,0.000000,0.000000
7,AdaBoost Tuned,uts_MPa,CrMo91,31,46.249806,77.184630,16.291828,36.505299,31,0,0.000000,0.000000
8,AdaBoost Tuned,elongation_pct,all,558,2.414214,3.026609,0.130173,0.126526,557,48,0.181818,0.104167
9,AdaBoost Tuned,elongation_pct,C-Mn,497,2.435078,3.045255,0.190162,0.194951,497,37,0.186047,0.108108


## 5. Histogram Gradient Boosting

We now evaluate Histogram Gradient Boosting as a third boosting method. As before, we first test the default model using the shared evaluation protocol before tuning its hyperparameters.

In [26]:
from sklearn.ensemble import HistGradientBoostingRegressor

hist_model = HistGradientBoostingRegressor(
    random_state=42
)

results_hist = evaluate(
    "HistGradientBoosting",
    hist_model,
    train
)

results_hist

,model,target,family,n,MAE,RMSE,MAE_std,RMSE_std,n_clf,n_nc,F1_nc,recall_nc
0,HistGradientBoosting,yield_strength_MPa,all,620,32.126318,45.526999,3.579121,7.607580,619,102,0.767568,0.696078
1,HistGradientBoosting,yield_strength_MPa,C-Mn,552,29.545571,39.830943,1.657942,3.174497,552,102,0.767568,0.696078
2,HistGradientBoosting,yield_strength_MPa,CrMo2,30,54.217226,72.919241,34.050349,39.549354,29,0,0.000000,0.000000
3,HistGradientBoosting,yield_strength_MPa,CrMo91,38,52.174874,81.080215,28.445184,41.981356,38,0,0.000000,0.000000
4,HistGradientBoosting,uts_MPa,all,596,27.863440,41.001458,3.119945,8.099273,595,198,0.738636,0.656566
5,HistGradientBoosting,uts_MPa,C-Mn,531,25.762002,34.499617,1.300682,2.236589,531,198,0.738636,0.656566
6,HistGradientBoosting,uts_MPa,CrMo2,34,48.759106,74.652425,26.508317,38.239324,33,0,0.000000,0.000000
7,HistGradientBoosting,uts_MPa,CrMo91,31,40.941213,76.296764,19.513120,46.178005,31,0,0.000000,0.000000
8,HistGradientBoosting,elongation_pct,all,558,1.976835,2.575766,0.187373,0.272827,557,48,0.575000,0.479167
9,HistGradientBoosting,elongation_pct,C-Mn,497,1.978231,2.571429,0.205537,0.284133,497,37,0.636364,0.567568


### 5.1 HistGradientBoosting Hyperparameter Tuning

The default Histogram Gradient Boosting model gives competitive results, particularly for Charpy toughness. We therefore tune its main hyperparameters using the same predefined cross-validation folds.

In [27]:
hist_param_grid = {
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_iter": [100, 200, 300],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__l2_regularization": [0.0, 0.1, 1.0]
}

In [28]:
hist_best_models = {}
hist_best_params = {}
hist_best_mae = {}

for target in TARGETS:
    X, y, cv = get_xy(train, target)

    model = HistGradientBoostingRegressor(
        random_state=42
    )

    pipeline = build_pipeline(model)

    grid_hist = GridSearchCV(
        estimator=pipeline,
        param_grid=hist_param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1
    )

    grid_hist.fit(X, y)

    hist_best_models[target] = grid_hist.best_estimator_
    hist_best_params[target] = grid_hist.best_params_
    hist_best_mae[target] = -grid_hist.best_score_

    print(f"\nTarget: {target}")
    print("Best parameters:", grid_hist.best_params_)
    print("Best CV MAE:", -grid_hist.best_score_)


Target: yield_strength_MPa
Best parameters: {'model__l2_regularization': 0.1, 'model__learning_rate': 0.1, 'model__max_iter': 300, 'model__max_leaf_nodes': 15}
Best CV MAE: 30.820195143840095

Target: uts_MPa
Best parameters: {'model__l2_regularization': 1.0, 'model__learning_rate': 0.1, 'model__max_iter': 300, 'model__max_leaf_nodes': 15}
Best CV MAE: 26.96723159891933

Target: elongation_pct
Best parameters: {'model__l2_regularization': 1.0, 'model__learning_rate': 0.1, 'model__max_iter': 100, 'model__max_leaf_nodes': 15}
Best CV MAE: 1.926161576658179

Target: charpy_toughness_J
Best parameters: {'model__l2_regularization': 1.0, 'model__learning_rate': 0.1, 'model__max_iter': 300, 'model__max_leaf_nodes': 31}
Best CV MAE: 18.66512589803815


In [29]:
hist_param_grid_refined = {
    "model__learning_rate": [0.1, 0.15, 0.2],
    "model__max_iter": [300, 400, 500],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__l2_regularization": [0.1, 1.0, 5.0]
}

In [30]:
hist_refined_best_models = {}
hist_refined_best_params = {}
hist_refined_best_mae = {}

for target in TARGETS:
    X, y, cv = get_xy(train, target)

    model = HistGradientBoostingRegressor(
        random_state=42
    )

    pipeline = build_pipeline(model)

    grid_hist_refined = GridSearchCV(
        estimator=pipeline,
        param_grid=hist_param_grid_refined,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1
    )

    grid_hist_refined.fit(X, y)

    hist_refined_best_models[target] = grid_hist_refined.best_estimator_
    hist_refined_best_params[target] = grid_hist_refined.best_params_
    hist_refined_best_mae[target] = -grid_hist_refined.best_score_

    print(f"\nTarget: {target}")
    print("Best parameters:", grid_hist_refined.best_params_)
    print("Best CV MAE:", -grid_hist_refined.best_score_)


Target: yield_strength_MPa
Best parameters: {'model__l2_regularization': 0.1, 'model__learning_rate': 0.1, 'model__max_iter': 500, 'model__max_leaf_nodes': 15}
Best CV MAE: 30.74393764119165

Target: uts_MPa
Best parameters: {'model__l2_regularization': 5.0, 'model__learning_rate': 0.1, 'model__max_iter': 400, 'model__max_leaf_nodes': 15}
Best CV MAE: 26.546456711448837

Target: elongation_pct
Best parameters: {'model__l2_regularization': 5.0, 'model__learning_rate': 0.1, 'model__max_iter': 300, 'model__max_leaf_nodes': 31}
Best CV MAE: 1.9518298275784525

Target: charpy_toughness_J
Best parameters: {'model__l2_regularization': 5.0, 'model__learning_rate': 0.1, 'model__max_iter': 500, 'model__max_leaf_nodes': 31}
Best CV MAE: 18.497663812165023


### 5.2 Final HistGradientBoosting Model Selection

The refined search produced small additional improvements for yield strength, UTS, and Charpy toughness, while the previous search remained better for elongation. Therefore, the best configuration from either search is retained separately for each target.

In [31]:
hist_final_models = {
    "yield_strength_MPa": hist_refined_best_models["yield_strength_MPa"],
    "uts_MPa": hist_refined_best_models["uts_MPa"],
    "elongation_pct": hist_best_models["elongation_pct"],
    "charpy_toughness_J": hist_refined_best_models["charpy_toughness_J"]
}

In [32]:
results_hist_final = evaluate(
    "HistGradientBoosting Tuned",
    hist_final_models,
    train
)

results_hist_final

,model,target,family,n,MAE,RMSE,MAE_std,RMSE_std,n_clf,n_nc,F1_nc,recall_nc
0,HistGradientBoosting Tuned,yield_strength_MPa,all,620,30.828974,45.128536,3.988589,8.058647,619,102,0.787234,0.725490
1,HistGradientBoosting Tuned,yield_strength_MPa,C-Mn,552,27.940137,38.157389,2.117045,3.182352,552,102,0.787234,0.725490
2,HistGradientBoosting Tuned,yield_strength_MPa,CrMo2,30,52.397129,76.576414,37.256696,41.537166,29,0,0.000000,0.000000
3,HistGradientBoosting Tuned,yield_strength_MPa,CrMo91,38,55.765640,86.307239,29.035427,42.866550,38,0,0.000000,0.000000
4,HistGradientBoosting Tuned,uts_MPa,all,596,26.578423,40.127234,3.174189,8.636222,595,198,0.770083,0.702020
5,HistGradientBoosting Tuned,uts_MPa,C-Mn,531,24.669795,33.341786,1.354908,2.303686,531,198,0.770083,0.702020
6,HistGradientBoosting Tuned,uts_MPa,CrMo2,34,45.147424,73.971233,29.909973,42.126797,33,0,0.000000,0.000000
7,HistGradientBoosting Tuned,uts_MPa,CrMo91,31,38.905372,76.903298,17.128917,44.757349,31,0,0.000000,0.000000
8,HistGradientBoosting Tuned,elongation_pct,all,558,1.927709,2.512208,0.147375,0.208364,557,48,0.589744,0.479167
9,HistGradientBoosting Tuned,elongation_pct,C-Mn,497,1.912601,2.487346,0.165068,0.235344,497,37,0.656250,0.567568


## 6. Comparison of Boosting Methods

The final tuned boosting models are compared using the common evaluation metrics. The comparison focuses mainly on the overall results (`family = all`) to identify the most effective boosting approach.

In [33]:
all_results = save_results(results_hist_final)

In [34]:
boosting_models = [
    "Gradient Boosting Tuned",
    "AdaBoost Tuned",
    "HistGradientBoosting Tuned"
]

boosting_comparison = all_results[
    (all_results["model"].isin(boosting_models)) &
    (all_results["family"] == "all")
]

boosting_comparison[
    [
        "model",
        "target",
        "MAE",
        "RMSE",
        "F1_nc",
        "recall_nc"
    ]
]

,model,target,MAE,RMSE,F1_nc,recall_nc
0,AdaBoost Tuned,yield_strength_MPa,43.794700,57.894200,0.019400,0.009800
4,AdaBoost Tuned,uts_MPa,40.214300,51.496800,0.579300,0.424200
8,AdaBoost Tuned,elongation_pct,2.414200,3.026600,0.181800,0.104200
12,AdaBoost Tuned,charpy_toughness_J,27.371900,35.735400,0.000000,0.000000
16,AdaBoost Tuned,label,NaN,NaN,0.306100,0.180700
80,Gradient Boosting Tuned,yield_strength_MPa,28.624800,41.533800,0.804300,0.725500
84,Gradient Boosting Tuned,uts_MPa,25.279700,40.828400,0.763000,0.666700
88,Gradient Boosting Tuned,elongation_pct,1.951200,2.553800,0.585400,0.500000
92,Gradient Boosting Tuned,charpy_toughness_J,19.609300,28.786200,0.300000,0.206900
96,Gradient Boosting Tuned,label,NaN,NaN,0.859000,0.807200


### 6.1 Boosting Model Selection

Among the tested boosting methods, AdaBoost showed substantially lower predictive performance and was therefore not retained as the preferred boosting approach.

Gradient Boosting and Histogram Gradient Boosting produced the strongest results. Gradient Boosting achieved the lowest MAE for yield strength and UTS, while Histogram Gradient Boosting performed best for elongation and Charpy toughness.

For the final conformity classification, Gradient Boosting achieved the best overall performance, with an F1-score of 0.859 and a recall of 0.807 for non-conforming welds. Histogram Gradient Boosting remained close, with an F1-score of 0.830 and a recall of 0.795.

Therefore, Gradient Boosting is retained as the best overall boosting model, while Histogram Gradient Boosting is particularly effective for the prediction of Charpy toughness.

## Conclusion

Among the tested boosting methods, Gradient Boosting provides the strongest overall performance. It achieves the best results for yield strength and UTS and the highest final F1-score and recall for detecting non-conforming welds.

Histogram Gradient Boosting performs slightly better for elongation and Charpy toughness and remains a strong alternative. AdaBoost is clearly less suitable for this dataset.

The boosting results can now be compared with the other supervised model families evaluated in the project.